<div style="padding: 20px; background: linear-gradient(90deg, #a70027ff 0%, #ff4b2b 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🌀 Module 7.1: RAG Fusion</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Combining Multi-Query generation with Mathematical Rank Fusion.</p>
</div>

---

## 1. What is RAG Fusion?

In **Module 6.3**, we built a Multi-Query Retriever that brainstormed 3 alternative queries, searched the DB, and glued the results together using a simple `set()`. 

**RAG Fusion** takes this to the professional level. Instead of a simple `set()`, it mathematically ranks the documents based on how often they appeared across the multiple searches, and how high they ranked in those searches. 

### The Reciprocal Rank Fusion (RRF) Formula:
$$ RRF(d) = \sum_{q \in Q} \frac{1}{k + rank_q(d)} $$

- $d$ is a Document.
- $Q$ is the list of multiple queries.
- $rank_q(d)$ is the position of the document in the search results for query $q$.
- $k$ is a smoothing constant (usually 60).

If a document shows up as Rank 1 across all 4 queries, its math score moons, and it becomes the absolute top result!

In [1]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from collections import defaultdict
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

docs = [
    Document(page_content="Quantum computing uses qubits that can exist in superposition."),
    Document(page_content="Quantum entanglement links qubits so measuring one instantly affects the other."),
    Document(page_content="Shor's algorithm on a quantum computer can factor large numbers exponentially faster."),
    Document(page_content="Current quantum computers suffer from decoherence and require near-absolute-zero temperatures."),
    Document(page_content="Classical computers use bits (0 or 1); quantum computers use qubits (0, 1, or both)."),
    Document(page_content="IBM and Google are leading quantum hardware manufacturers."),
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = Chroma.from_documents(docs, embeddings, collection_name="fusion_demo")
print("Database loaded with Quantum Computing facts.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database loaded with Quantum Computing facts.


## 2. Generating the Queries
We use Groq to quickly generate multiple variations of the user's question.

In [2]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)
    
    query_gen_prompt = ChatPromptTemplate.from_template("""
    Generate {n} different search queries for: "{question}"
    Output only the queries, one per line, without any numbering or bullet points.
    """)
    
    def generate_queries(question: str, n: int = 4) -> list[str]:
        chain = query_gen_prompt | llm | StrOutputParser()
        result = chain.invoke({"question": question, "n": n})
        queries = [q.strip() for q in result.strip().split("\n") if q.strip()]
        return queries[:n]
else:
    print("GROQ_API_KEY missing. Please add to .env file.")

## 3. The Reciprocal Rank Fusion Math
We search the Vector DB for *each* generated query, and then run the RRF math on the lists to find the ultimate winners.

In [3]:
if groq_api_key:
    def reciprocal_rank_fusion(results_lists: list[list[Document]], k: int = 60) -> list[Document]:
        scores = defaultdict(float)
        doc_map = {}
    
        for ranked_list in results_lists:
            for rank, doc in enumerate(ranked_list, 1):
                key = doc.page_content
                scores[key] += 1.0 / (k + rank)
                doc_map[key] = doc
    
        sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_map[key] for key, _ in sorted_docs]
    
    original_query = "How do quantum computers work?"
    variants = generate_queries(original_query, n=3)
    all_queries = [original_query] + variants
    
    print("--- GENERATED QUERIES ---")
    for q in all_queries:
        print(f" • {q}")
    
    # 1. Search the DB for every single query variant
    retrieved_lists = [vs.similarity_search(q, k=4) for q in all_queries]
    
    # 2. Mathematically fuse the lists together!
    fused_docs = reciprocal_rank_fusion(retrieved_lists)
    
    print(f"\n--- RAG FUSION RESULTS ({len(fused_docs)} unique docs) ---")
    for i, d in enumerate(fused_docs, 1):
        print(f"[{i}] {d.page_content}")

--- GENERATED QUERIES ---
 • How do quantum computers work?
 • What is the fundamental difference between classical and quantum computing?
 • How do quantum bits or qubits enable quantum computing?
 • What are the key principles behind quantum computing and its applications?



--- RAG FUSION RESULTS (6 unique docs) ---
[1] Quantum computing uses qubits that can exist in superposition.
[2] Classical computers use bits (0 or 1); quantum computers use qubits (0, 1, or both).
[3] Shor's algorithm on a quantum computer can factor large numbers exponentially faster.
[4] IBM and Google are leading quantum hardware manufacturers.
[5] Current quantum computers suffer from decoherence and require near-absolute-zero temperatures.
[6] Quantum entanglement links qubits so measuring one instantly affects the other.
